# 01 — Build odds and scores

Build 2025-26 NBA regular-season dataset: pregame (closing) odds + final scores.

Sources
- Scores + game list: Basketball-Reference monthly schedule pages
  (bref/*.html, already fetched). Regular season = date <= 2026-04-12,
  excluding the 2025-12-16 NBA Cup final (SAS @ NYK), which is not a
  regular-season game. That yields exactly 1,230 games (30 teams x 82 / 2).
- Odds: TeamRankings matchup pages (server-rendered, polite 2s delay).
  * Spread + total: matchup page header, e.g. "Odds: Oklahoma City by 6.5,
    Total Points: 226". This is the closing line (matches the line-movement
    pages' closing values on sampled games).
  * Moneyline: matchup money-line-movement page, closing ("Current") values
    from the ps-team cells, e.g. OKC -260 / HOU +215.

OddsPortal was the original primary source, but its results pagination is
client-side only: /results/page/N returns a shell and the RSC navigation
requests hang indefinitely (5 attempts, 0 bytes, incl. with cookies and
browser-style headers). Only page 1 (50 playoff games) is server-rendered.
TeamRankings is therefore the odds source; its lines are the closing lines
shown on each game's recap page.

Output: nba_2025_26_odds_scores.csv (one row per game).
Checkpointing: rows are appended as fetched; re-running resumes from the
existing CSV. Failures are logged to failures.json for a second pass.


In [ ]:
import csv
import json
import os
import re
import time
import urllib.request

from pathlib import Path
# project root: works whether this runs from notebooks/ or the project root
def _find_proj(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / 'data' / 'raw' / 'nba_2025_26_odds_scores.csv').exists():
            return c
        if (c / 'data' / 'raw').is_dir() and (c / 'notebooks').is_dir():
            return c
    return p
PROJ = _find_proj(Path.cwd())
RAW = PROJ / 'data' / 'raw'
INTERIM = PROJ / 'data' / 'interim'
OUT = PROJ / 'outputs'
RES = PROJ / 'results'
CACHE = PROJ / 'cache'

SLUGS = {
    'Atlanta Hawks': 'hawks', 'Boston Celtics': 'celtics', 'Brooklyn Nets': 'nets',
    'Charlotte Hornets': 'hornets', 'Chicago Bulls': 'bulls', 'Cleveland Cavaliers': 'cavaliers',
    'Dallas Mavericks': 'mavericks', 'Denver Nuggets': 'nuggets', 'Detroit Pistons': 'pistons',
    'Golden State Warriors': 'warriors', 'Houston Rockets': 'rockets', 'Indiana Pacers': 'pacers',
    'Los Angeles Clippers': 'clippers', 'Los Angeles Lakers': 'lakers', 'Memphis Grizzlies': 'grizzlies',
    'Miami Heat': 'heat', 'Milwaukee Bucks': 'bucks', 'Minnesota Timberwolves': 'timberwolves',
    'New Orleans Pelicans': 'pelicans', 'New York Knicks': 'knicks',
    'Oklahoma City Thunder': 'thunder', 'Orlando Magic': 'magic',
    'Philadelphia 76ers': '76ers', 'Phoenix Suns': 'suns',
    'Portland Trail Blazers': 'trail-blazers', 'Sacramento Kings': 'kings',
    'San Antonio Spurs': 'spurs', 'Toronto Raptors': 'raptors', 'Utah Jazz': 'jazz',
    'Washington Wizards': 'wizards',
}
BASE = 'https://www.teamrankings.com/nba/matchup/'
def matchup_url(away, home, date):
    return f"{BASE}{SLUGS[away]}-{SLUGS[home]}-{date}/"

OUT_CSV = RAW / 'nba_2025_26_odds_scores.csv'
FAIL_LOG = CACHE / 'failures.json'
STATE = CACHE / 'build_state.json'

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
DELAY = 2.0

ABBREV = {
    "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets",
    "CHA": "Charlotte Hornets", "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets", "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat", "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors", "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
}

# short names the TeamRankings "Odds: X by N" header may use, per team
CITY = {
    "Atlanta Hawks": ["atlanta"], "Boston Celtics": ["boston"],
    "Brooklyn Nets": ["brooklyn"], "Charlotte Hornets": ["charlotte"],
    "Chicago Bulls": ["chicago"], "Cleveland Cavaliers": ["cleveland"],
    "Dallas Mavericks": ["dallas"], "Denver Nuggets": ["denver"],
    "Detroit Pistons": ["detroit"], "Golden State Warriors": ["golden state"],
    "Houston Rockets": ["houston"], "Indiana Pacers": ["indiana"],
    "Los Angeles Clippers": ["los angeles", "la clippers"],
    "Los Angeles Lakers": ["los angeles", "la lakers"],
    "Memphis Grizzlies": ["memphis"], "Miami Heat": ["miami"],
    "Milwaukee Bucks": ["milwaukee"], "Minnesota Timberwolves": ["minnesota"],
    "New Orleans Pelicans": ["new orleans"], "New York Knicks": ["new york"],
    "Oklahoma City Thunder": ["oklahoma city", "okla city"],
    "Orlando Magic": ["orlando"], "Philadelphia 76ers": ["philadelphia"],
    "Phoenix Suns": ["phoenix"], "Portland Trail Blazers": ["portland"],
    "Sacramento Kings": ["sacramento"], "San Antonio Spurs": ["san antonio"],
    "Toronto Raptors": ["toronto"], "Utah Jazz": ["utah"],
    "Washington Wizards": ["washington"],
}
MASCOT = {
    "Atlanta Hawks": "hawks", "Boston Celtics": "celtics", "Brooklyn Nets": "nets",
    "Charlotte Hornets": "hornets", "Chicago Bulls": "bulls",
    "Cleveland Cavaliers": "cavaliers", "Dallas Mavericks": "mavericks",
    "Denver Nuggets": "nuggets", "Detroit Pistons": "pistons",
    "Golden State Warriors": "warriors", "Houston Rockets": "rockets",
    "Indiana Pacers": "pacers", "Los Angeles Clippers": "clippers",
    "Los Angeles Lakers": "lakers", "Memphis Grizzlies": "grizzlies",
    "Miami Heat": "heat", "Milwaukee Bucks": "bucks",
    "Minnesota Timberwolves": "timberwolves", "New Orleans Pelicans": "pelicans",
    "New York Knicks": "knicks", "Oklahoma City Thunder": "thunder",
    "Orlando Magic": "magic", "Philadelphia 76ers": "76ers",
    "Phoenix Suns": "suns", "Portland Trail Blazers": "trail blazers",
    "Sacramento Kings": "kings", "San Antonio Spurs": "spurs",
    "Toronto Raptors": "raptors", "Utah Jazz": "jazz",
    "Washington Wizards": "wizards",
}


In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", s.strip().lower())


def fetch(url, timeout=30):
    """Fetch with retries: transient timeouts/resets are common on
    TeamRankings after many requests; retry with backoff before giving up."""
    last = None
    for attempt, wait in enumerate((0, 5)):
        if wait:
            time.sleep(wait)
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read().decode("utf-8", "replace")
        except Exception as e:
            last = e
            print(f"  retry {attempt + 1}/2 {url[:80]}... {e!r}", flush=True)
    raise last


def parse_main(h):
    """Return (fav_header_name, spread_points, total) from matchup page.
    TeamRankings sometimes injects an <svg> 'suspect line' icon between the
    spread number and the comma (e.g. huge 23.5-pt lines), so strip tags
    from the Odds header before matching."""
    hm = re.search(r"Odds:.*?(?:Total Points:\s*[\d.]+)", h, re.S)
    if not hm:
        return None
    clean = re.sub(r"<[^>]+>", " ", hm.group(0))
    m = re.search(r"Odds:\s*(.+?)\s+by\s+([\d.]+)\s*,\s+Total Points:\s+([\d.]+)", clean, re.S)
    if not m:
        return None
    return norm(m.group(1)), float(m.group(2)), float(m.group(3))


def parse_ml(h):
    """Return {full_team_name: american_ml} from the ML movement page."""
    out = {}
    for abbr, val in re.findall(
            r'<td class="ps-team"[^>]*><a[^>]*>([A-Z]{2,3})\s*([+-]\d+)</a></td>', h):
        if abbr in ABBREV:
            out[ABBREV[abbr]] = int(val)
    if len(out) < 2:
        # fallback: history table "Current" row with team-abbr column headers
        hm = re.search(
            r'<td>Current</td>\s*<td[^>]*>([+-]\d+)</td>\s*<td[^>]*>([+-]\d+)</td>', h)
        th = re.findall(r'<th[^>]*>([A-Z]{2,3})</th>', h)
        if hm and len(th) >= 2:
            for abbr, val in zip(th[:2], hm.groups()):
                if abbr in ABBREV:
                    out.setdefault(ABBREV[abbr], int(val))
    return out


def fav_team(header_name, away, home, ml):
    """Resolve the header's favorite name to one of the two teams."""
    cands = [t for t in (away, home)
             if header_name in CITY.get(t, []) or header_name == MASCOT.get(t)]
    if len(cands) == 1:
        return cands[0]
    if len(cands) > 1:
        # ambiguous (both LA teams): use the moneyline favorite
        neg = [t for t, v in ml.items() if v < 0]
        if len(neg) == 1 and neg[0] in cands:
            return neg[0]
        return None
    return None


def load_games():
    games = json.load(open(CACHE / 'bref_games.json'))
    rs = [g for g in games
          if g["date"] <= "2026-04-12"
          and not (g["date"] == "2025-12-16"
                   and {g["away"], g["home"]} == {"San Antonio Spurs", "New York Knicks"})]
    rs.sort(key=lambda g: (g["date"], g["away"], g["home"]))
    return rs


def done_keys():
    keys = set()
    if os.path.exists(OUT_CSV):
        with open(OUT_CSV, newline="") as f:
            for row in csv.DictReader(f):
                keys.add((row["date"], row["away_team"], row["home_team"]))
    return keys


In [ ]:
def main():
    games = load_games()
    print(f"regular-season games to build: {len(games)}", flush=True)
    done = done_keys()
    print(f"already in CSV: {len(done)}", flush=True)
    failures = []
    if os.path.exists(FAIL_LOG):
        failures = json.load(open(FAIL_LOG))
    new_file = not os.path.exists(OUT_CSV)
    with open(OUT_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["date", "away_team", "home_team", "away_score", "home_score",
                        "home_ml", "away_ml", "home_spread", "total", "notes"])
        for i, g in enumerate(games):
            key = (g["date"], g["away"], g["home"])
            if key in done:
                continue
            notes = []
            home_ml = away_ml = home_spread = total = ""
            try:
                base = matchup_url(g["away"], g["home"], g["date"])
                main_h = fetch(base)
                time.sleep(DELAY)
                parsed = parse_main(main_h)
                if parsed is None:
                    raise ValueError("no Odds header on matchup page")
                fav_name, pts, total = parsed
                ml_h = fetch(base + "money-line-movement")
                time.sleep(DELAY)
                ml = parse_ml(ml_h)
                if g["home"] in ml:
                    home_ml = ml[g["home"]]
                if g["away"] in ml:
                    away_ml = ml[g["away"]]
                if not home_ml or not away_ml:
                    notes.append("ML missing for one/both teams on TeamRankings ML-movement page")
                fav = fav_team(fav_name, g["away"], g["home"], ml)
                if fav is None:
                    notes.append(f"spread favorite unresolvable (header: {fav_name!r})")
                    home_spread = ""
                else:
                    home_spread = -pts if fav == g["home"] else pts
                    # sanity: ML favorite should match spread favorite
                    neg = [t for t, v in ml.items() if v < 0]
                    if len(neg) == 1 and neg[0] != fav:
                        notes.append("ML favorite disagrees with spread favorite")
            except Exception as e:
                notes.append(f"fetch/parse failed: {e!r}")
                failures.append({"date": g["date"], "away": g["away"], "home": g["home"],
                                 "error": repr(e)})
                json.dump(failures, open(FAIL_LOG, "w"), indent=1)
                time.sleep(DELAY)
            row = [g["date"], g["away"], g["home"], g["away_pts"], g["home_pts"],
                   home_ml, away_ml,
                   (f"{home_spread:g}" if home_spread != "" else ""),
                   (f"{total:g}" if total != "" else ""),
                   "; ".join(notes) if notes else
                   "odds: TeamRankings closing (matchup header + ML movement); scores: Basketball-Reference"]
            w.writerow(row)
            f.flush()
            done.add(key)
            n = len(done)
            if n % 50 == 0:
                print(f"{n}/{len(games)} done, failures so far: {len(failures)}", flush=True)
    print(f"DONE. rows in CSV: {len(done)}, failures: {len(failures)}", flush=True)


In [ ]:
main()


## Retry pass

Second pass: re-fetch odds for rows in nba_2025_26_odds_scores.csv that
have any missing odds field (from transient TeamRankings timeouts during
the main build). Reuses the parsing/fetch helpers defined above in this notebook.
Rewrites the CSV in place; only fills fields that are blank.

The helpers above (`fetch`, `parse_main`, `parse_ml`, `fav_team`) are already defined in this notebook, so the retry runs inline. Only blank fields are filled.


In [ ]:
rows = list(csv.DictReader(open(OUT_CSV)))
fields = rows[0].keys() if rows else []
todo = [r for r in rows
        if not r["home_ml"] or not r["away_ml"] or not r["home_spread"] or not r["total"]]
print(f"incomplete rows to retry: {len(todo)}", flush=True)
fixed = still = 0
for r in todo:
    try:
        base = matchup_url(r["away_team"], r["home_team"], r["date"])
        main_h = fetch(base)
        time.sleep(DELAY)
        parsed = parse_main(main_h)
        if parsed is None:
            raise ValueError("no Odds header on matchup page")
        fav_name, pts, total = parsed
        ml_h = fetch(base + "money-line-movement")
        time.sleep(DELAY)
        ml = parse_ml(ml_h)
        if not r["home_ml"] and r["home_team"] in ml:
            r["home_ml"] = ml[r["home_team"]]
        if not r["away_ml"] and r["away_team"] in ml:
            r["away_ml"] = ml[r["away_team"]]
        if not r["home_spread"]:
            fav = fav_team(fav_name, r["away_team"], r["home_team"], ml)
            if fav is not None:
                r["home_spread"] = f"{(-pts if fav == r['home_team'] else pts):g}"
        if not r["total"]:
            r["total"] = f"{total:g}"
        if r["home_ml"] and r["away_ml"] and r["home_spread"] and r["total"]:
            r["notes"] = ("odds: TeamRankings closing (matchup header + ML movement); "
                          "scores: Basketball-Reference")
            fixed += 1
            print(f"FIXED {r['date']} {r['away_team']} @ {r['home_team']}", flush=True)
        else:
            r["notes"] = "still incomplete after retry pass"
            still += 1
            print(f"STILL MISSING {r['date']} {r['away_team']} @ {r['home_team']}", flush=True)
    except Exception as e:
        r["notes"] = f"retry failed: {e!r}"
        still += 1
        print(f"RETRY FAILED {r['date']} {r['away_team']} @ {r['home_team']}: {e!r}", flush=True)
        time.sleep(DELAY)

with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(fields))
    w.writeheader()
    w.writerows(rows)
print(f"DONE. fixed: {fixed}, still incomplete: {still}", flush=True)
